In [ ]:
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree
import requests
import time
import os
import numpy as np
!pip install python-dotenv
from dotenv import load_dotenv

load_dotenv()

In [ ]:
onemap_api_key = os.environ.get('ONEMAP_API_KEY')

if not onemap_api_key:
    raise ValueError(
        "ONEMAP_API_KEY not found. Please create a .env file in this directory with: \n"
        "ONEMAP_API_KEY=your_api_key_here"
    )

In [ ]:
# Get file names 
hdb_points_file = "resale_2019_to_2025_cleaned.geojson"
mrt_exits_file = "LTAMRTStationExitGEOJSON.geojson"
output_file = "hdb_mrt_distances.csv"

### Run a sample of postal codes to ensure code works, check the result output to see if there are any nulls 

In [ ]:
# --- 1. load and prep data ---
raw_hdb_gdf = gpd.read_file(hdb_points_file).to_crs(epsg=4326)
# take only the first 5 unique postals for this sample
postal_gdf_sample = raw_hdb_gdf.drop_duplicates(subset=['postal_code']).head(5).copy()

mrt_exits_gdf = gpd.read_file(mrt_exits_file).to_crs(epsg=4326)

mrt_coords = list(zip(mrt_exits_gdf.geometry.y, mrt_exits_gdf.geometry.x))
mrt_names = mrt_exits_gdf['STATION_NA'].tolist() 
mrt_exit_codes = mrt_exits_gdf['EXIT_CODE'].tolist() 
tree = cKDTree(mrt_coords)

results = []

print(f"--- starting sample test for {len(postal_gdf_sample)} postals ---")

# --- 2. sample loop ---
for idx, row in postal_gdf_sample.iterrows():
    p_code = str(row['postal_code'])
    origin_coords = (row.geometry.y, row.geometry.x)
    
    print(f"\nchecking postal: {p_code} at {origin_coords}")
    
    # get 3 closest mrt exits by straight line
    dist, indices = tree.query(origin_coords, k=3) 
    
    best_walk = float('inf')
    closest_mrt = None
    closest_exit = None

    for i in indices:
        dest_coords = mrt_coords[i]
        mrt_name = mrt_names[i]
        mrt_exit = mrt_exit_codes[i]
        
        # using the routingsvc endpoint from the docs
        url = f"https://www.onemap.gov.sg/api/public/routingsvc/route?start={origin_coords[0]},{origin_coords[1]}&end={dest_coords[0]},{dest_coords[1]}&routeType=walk"
        
        try:

            token = onemap_api_key if onemap_api_key.startswith("Bearer ") else f"Bearer {onemap_api_key}"
            headers = {"Authorization": token}
            res = requests.get(url, headers=headers, timeout=10)
            
            if res.status_code == 200:
                data = res.json()
                if 'route_summary' in data:
                    walk_m = data['route_summary']['total_distance']
                    print(f"  -> found route to {mrt_name}: {walk_m}m")
                    if walk_m < best_walk:
                        best_walk = walk_m
                        closest_mrt = mrt_name
                        closest_exit = mrt_exit
                else:
                    print(f"  -> no route summary found for {mrt_name}")
            else:
                print(f"  -> api error {res.status_code}: {res.text}")
            
            time.sleep(0.2) 
        except Exception as e:
            print(f"  -> request failed: {e}")

    results.append({
        "postal_code": p_code,
        "nearest_mrt": closest_mrt,
        "nearest_mrt_exit": closest_exit,
        "walking_dist_m": best_walk if best_walk != float('inf') else None
    })

# --- 3. display output ---
sample_df = pd.DataFrame(results)
print("\n--- final sample output ---")
print(sample_df)

# check if we actually got results
if sample_df['walking_dist_m'].isnull().all():
    print("\nwarning: all walking distances are null. check your api key and coordinate order.")
else:
    print("\nsuccess: distances found! you can now run the full script.")

### Run full script if results generated for the sample postal codes

In [ ]:
# --- 1. Load Datasets (full) ---
raw_hdb_gdf = gpd.read_file(hdb_points_file).to_crs(epsg=4326)
postal_gdf = raw_hdb_gdf.drop_duplicates(subset=['postal_code']).copy()

mrt_exits_gdf = gpd.read_file(mrt_exits_file).to_crs(epsg=4326)

mrt_coords = list(zip(mrt_exits_gdf.geometry.y, mrt_exits_gdf.geometry.x))
mrt_names = mrt_exits_gdf['STATION_NA'].tolist() 
mrt_exit_codes = mrt_exits_gdf['EXIT_CODE'].tolist() 
tree = cKDTree(mrt_coords)

# --- 2. Resume logic ---
if os.path.exists(output_file):
    processed_df = pd.read_csv(output_file)
    processed_postals = set(processed_df['postal_code'].astype(str))
    results = processed_df.to_dict('records')
    print(f"Resuming... {len(processed_postals)} postals already completed.")
else:
    results = []
    processed_postals = set()

# --- 3. Full loop ---
print(f"Starting processing for {len(postal_gdf) - len(processed_postals)} remaining postals...")

try:
    for idx, row in postal_gdf.iterrows():
        p_code = row['postal_code']
        if p_code in processed_postals:
            continue

        origin_coords = (row.geometry.y, row.geometry.x)
        dist, indices = tree.query(origin_coords, k=3)
        
        best_walk = float('inf')
        best_mrt = None
        best_mrt_exit = None

        for i in indices:
            dest_coords = mrt_coords[i]
            token = onemap_api_key if onemap_api_key.startswith("Bearer ") else f"Bearer {onemap_api_key}"
            url = f"https://www.onemap.gov.sg/api/public/routingsvc/route?start={origin_coords[0]},{origin_coords[1]}&end={dest_coords[0]},{dest_coords[1]}&routeType=walk"
            
            try:
                res = requests.get(url, headers={"Authorization": token}, timeout=10)
                if res.status_code == 200:
                    data = res.json()
                    if 'route_summary' in data:
                        walk_m = data['route_summary']['total_distance']
                        if walk_m < best_walk:
                            best_walk = walk_m
                            best_mrt = mrt_names[i]
                            best_mrt_exit = mrt_exit_codes[i]
                time.sleep(0.2)
            except:
                continue

        results.append({
            "postal_code": p_code,
            "nearest_mrt": best_mrt,
            "nearest_exit": best_mrt_exit,
            "walking_dist_m": best_walk if best_walk != float('inf') else None
        })
        processed_postals.add(p_code)

        if len(results) % 50 == 0:
            pd.DataFrame(results).to_csv(output_file, index=False)
            print(f"progress: {len(results)} / {len(postal_gdf)} unique postals saved.")
            

except KeyboardInterrupt:
    print("\nPaused. Saving current progress...")

# Final save
pd.DataFrame(results).to_csv(output_file, index=False)
print(f"Task complete! Results saved to {output_file}")

In [ ]:
hdb_mrt_distances = pd.read_csv("hdb_mrt_distances.csv")

### Checking for any anomalies, then verify on onemap if walking distance is too big

In [ ]:
hdb_mrt_distances[hdb_mrt_distances["walking_dist_m"] > 2500]

### Get feature variables of mrt 
- num_unique_mrts_500m
- num_unique_mrts_1km
- num_mrt_lines_500m
- num_mrt_lines_1km

In [ ]:
# read files
unique_hdb = gpd.read_file(hdb_points_file).copy()
mrt_exits_gdf = gpd.read_file(mrt_exits_file).copy()
hdb_mrt_distances = pd.read_csv("hdb_mrt_distances.csv")

# get unique hdb locations
unique_hdb = unique_hdb.drop_duplicates(subset=["postal_code"]).copy()
unique_hdb = unique_hdb.to_crs(epsg=3414)
mrt_exits_gdf = mrt_exits_gdf.to_crs(epsg=3414)

# prepare unique hdb and mrt exit coordinates for distance calculation
hdb_coords = np.array([(geom.x, geom.y) for geom in unique_hdb.geometry])
mrt_exits_coords = np.array([(geom.x, geom.y) for geom in mrt_exits_gdf.geometry])

mrt_names = mrt_exits_gdf["STATION_NA"].astype(str).tolist()

# build cKDTree for efficient spatial queries
tree = cKDTree(mrt_exits_coords)

# get MRT stations within 500m and 1km
indices_500m = tree.query_ball_point(hdb_coords, r=500)
indices_1km = tree.query_ball_point(hdb_coords, r=1000)

def clean_station_list(stations):
    # standardise station codes/names 
    name_map = {
        "CC9": "PAYA LEBAR MRT STATION",
        "DT18": "TELOK AYER MRT STATION"
    }
    stations = [name_map.get(s, s) for s in stations]

    # remove duplicates while preserving order
    stations = list(dict.fromkeys(stations))
    station_set = set(stations)

    # if both MRT and LRT with same station name are present, keep MRT only
    replacements = [
        ("BUKIT PANJANG MRT STATION", "BUKIT PANJANG LRT STATION"),
        ("PUNGGOL MRT STATION", "PUNGGOL LRT STATION"),
        ("CHOA CHU KANG MRT STATION", "CHOA CHU KANG LRT STATION"),
    ]

    for mrt_name, lrt_name in replacements:
        if mrt_name in station_set and lrt_name in station_set:
            stations = [s for s in stations if s != lrt_name]

    return stations

# convert index arrays into mrt station lists
unique_hdb["mrt_stations_within_500m"] = [
    clean_station_list([mrt_names[i] for i in idx_list])
    for idx_list in indices_500m
]

unique_hdb["mrt_stations_within_1km"] = [
    clean_station_list([mrt_names[i] for i in idx_list])
    for idx_list in indices_1km
]

# count columns
unique_hdb["num_unique_stations_500m"] = unique_hdb["mrt_stations_within_500m"].apply(len)
unique_hdb["num_unique_stations_1km"] = unique_hdb["mrt_stations_within_1km"].apply(len)

mrt_station_counts = unique_hdb[[
    "postal_code",
    "mrt_stations_within_500m",
    "mrt_stations_within_1km",
    "num_unique_stations_500m",
    "num_unique_stations_1km"
]].copy()

print(mrt_station_counts)

In [ ]:
mrt_station_counts.to_csv("station_counts.csv", index = False)

mrt_station_counts.to_excel("station_counts.xlsx")

In [ ]:
## need merge with dataset containing number of lines per station, then merge with mrt-walking dataset
import ast
mrt_stations_masterlist = pd.read_excel("mrt_stations.xlsx")

In [ ]:

# clean lookup table
mrt_stations_masterlist["STN_NAM"] = mrt_stations_masterlist["STN_NAM"].astype(str).str.strip()
mrt_stations_masterlist["LINES_CONNECTED"] = mrt_stations_masterlist["LINES_CONNECTED"].astype(str).str.strip()

# build dictionary: station -> list of lines
station_to_lines = {
    row["STN_NAM"]: [x.strip() for x in row["LINES_CONNECTED"].split(",")]
    for _, row in mrt_stations_masterlist.iterrows()
}

def parse_station_list(station_list):
    # handles cases where list was saved/read as string from csv/excel
    if isinstance(station_list, str):
        station_list = ast.literal_eval(station_list)
    return [str(s).strip() for s in station_list]

def get_unique_lines(station_list):
    station_list = parse_station_list(station_list)
    unique_lines = set()

    for station in station_list:
        lines = station_to_lines.get(station, [])
        unique_lines.update(lines)

    return sorted(unique_lines)

# apply to existing mrt_station_counts dataframe
mrt_station_counts["unique_lines_500m"] = mrt_station_counts["mrt_stations_within_500m"].apply(get_unique_lines)
mrt_station_counts["unique_lines_1km"] = mrt_station_counts["mrt_stations_within_1km"].apply(get_unique_lines)

mrt_station_counts["num_unique_mrt_lines_500m"] = mrt_station_counts["unique_lines_500m"].apply(len)
mrt_station_counts["num_unique_mrt_lines_1km"] = mrt_station_counts["unique_lines_1km"].apply(len)


print(mrt_station_counts.head())

In [ ]:
mrt_station_counts.head(1)
mrt_station_counts.info()

In [ ]:
hdb_mrt_distances.head(1)
hdb_mrt_distances.info()


In [ ]:
# Final output 
mrt_station_counts["postal_code"] = mrt_station_counts["postal_code"].astype(str).str.strip().str.zfill(6)
hdb_mrt_distances["postal_code"] = hdb_mrt_distances["postal_code"].astype(str).str.strip().str.zfill(6)


merged_dataset = mrt_station_counts.merge(hdb_mrt_distances, on = "postal_code", how = "inner")

In [ ]:
merged_dataset.head(3)


In [ ]:
final = merged_dataset[["postal_code",
                        "num_unique_mrt_500m", 
                        "num_unique_mrt_1km",
                        "num_unique_mrt_lines_500m", 
                        "num_unique_mrt_lines_1km",
                        "walking_dist_m"
                        ]]

final = final.rename(columns = {"walking_dist_m":"walking_dist_mrt_m"})

In [ ]:
final.head(3)

In [ ]:
final.to_csv("hdb_mrt_distances.csv", index = False)